# PRGI Title Verification — Hackathon Demo
> **Press Registrar General of India** · Innovation Hackathon 2025

This notebook demonstrates the full verification pipeline:
1. Preprocessing & normalisation
2. Phonetic matching (Soundex / Metaphone / NYSIIS)
3. Fuzzy string similarity (Levenshtein · Jaro-Winkler · n-gram)
4. Semantic embedding comparison (multilingual SBERT + FAISS)
5. Cross-language translation check
6. Compliance rules (banned words · periodicity · combinations)
7. Score aggregation → APPROVED / REVIEW / REJECTED

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # make sure pipeline/ is importable

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pipeline.preprocessor   import preprocess, strip_affixes
from pipeline.phonetic        import phonetic_score, phonetic_match
from pipeline.fuzzy           import combined_fuzzy_score, top_similar
from pipeline.cross_language  import cross_language_check
from pipeline.compliance      import check_disallowed, check_periodicity, check_combination

# Load titles DB
df = pd.read_csv('data/titles.csv')
TITLES = df.iloc[:, 0].dropna().astype(str).tolist()
print(f'Loaded {len(TITLES)} titles')

## 1 · Preprocessing

In [ ]:
test_titles = [
    'Surya Bharath',
    'Hindustan Morning Times Daily',
    'Delhi Police Chronicle',
    'Prabhat Samachar',
    'Quantum Digest',
]

for t in test_titles:
    p = preprocess(t)
    print(f"  {t!r:35s}  →  normalized={p['normalized']!r}  stripped={p['stripped']!r}")

## 2 · Phonetic Matching

In [ ]:
pairs = [
    ('Namaskar', 'Namascar'),
    ('Hindu',    'Hindoo'),
    ('Surya',    'Suria'),
    ('Pratidin', 'Pratideen'),
    ('Tribune',  'Deccan'),
]

print(f"{'Word 1':<16} {'Word 2':<16} {'Score':>6}  Match")
print('-' * 52)
for w1, w2 in pairs:
    sc = phonetic_score(w1, w2)
    print(f"{w1:<16} {w2:<16} {sc:>6.2f}  {'✓' if sc >= 0.67 else '✗'}")

## 3 · Fuzzy String Similarity

In [ ]:
query = 'Morning Herald'
results = top_similar(strip_affixes(query), [strip_affixes(t) for t in TITLES], top_k=5)

print(f"Top 5 fuzzy matches for: '{query}'\n")
print(f"{'Title':<35} {'Score':>6}")
print('-' * 44)
for r in results:
    print(f"{r['title']:<35} {r['score']:>6.1f}")

## 4 · Semantic Search  
> **Requires index to be built first:**  `python scripts/build_index.py`

In [ ]:
from pipeline.semantic import semantic_search

query = 'Subah Ka Samachar'
results = semantic_search(query, top_k=5)

if results:
    print(f"Semantic matches for: '{query}'\n")
    for r in results:
        print(f"  {r['title']:<35}  {r['score']:.1f}%")
else:
    print('[SKIP] FAISS index not built. Run: python scripts/build_index.py')

## 5 · Cross-Language Check

In [ ]:
cases = [
    ['prabhat', 'samachar', 'india'],   # prabhat = morning  →  conflicts with 'Morning Herald'
    ['surya',   'times'],               # surya   = sun      →  conflicts with 'Sun Times' etc.
    ['quantum', 'digest'],              # no cross-lang match
]

for tokens in cases:
    result = cross_language_check(tokens, TITLES)
    icon   = '✗ FAIL' if result['status'] == 'fail' else '✓ PASS'
    print(f"{icon}  tokens={tokens}")
    if result['issue']:
        print(f"       → {result['issue']['detail']}")

## 6 · Compliance Rules

In [ ]:
test_cases = [
    ('Delhi Police Chronicle', ''),
    ('Morning Herald Daily',   'daily'),
    ('Hindustan Times Weekly', 'weekly'),
    ('Navbharat Samachar',     ''),
]

from pipeline.preprocessor import tokenize

for title, period in test_cases:
    tokens = tokenize(title)
    d = check_disallowed(tokens)
    p = check_periodicity(tokens, period, TITLES)
    c = check_combination(tokens, TITLES)
    issues = [x for x in [d, p, c] if x['status'] == 'fail']
    status = 'FAIL' if issues else 'PASS'
    print(f"  [{status}] {title!r}")
    for iss in issues:
        print(f"          → {iss['issue']['detail']}")

## 7 · End-to-End via REST API
> **Start the server first:**  `uvicorn main:app --reload --port 8000`

In [ ]:
import requests, json

BASE = 'http://localhost:8000'

test_payloads = [
    {'title': 'Surya Bharath',              'periodicity': ''},
    {'title': 'Delhi Police Times',         'periodicity': ''},
    {'title': 'Morning Herald Daily',       'periodicity': 'daily'},
    {'title': 'Hindustan Times Weekly',     'periodicity': 'weekly'},
    {'title': 'Quantum Digest',             'periodicity': ''},
    {'title': 'Prabhat Samachar',           'periodicity': ''},
]

print(f"{'Title':<35} {'Verdict':<10} {'Prob':>5}  {'Max Sim':>8}")
print('-' * 65)
for payload in test_payloads:
    try:
        r = requests.post(f'{BASE}/verify', json=payload, timeout=10)
        d = r.json()
        print(f"{payload['title']:<35} {d['verdict']:<10} {d['verify_prob']:>4}%  {d['max_sim']:>7.1f}%")
    except Exception as e:
        print(f"[ERROR] {e} — is the server running?")
        break

## 8 · Score Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_titles = [
    'Surya Bharath', 'Quantum Digest', 'Morning Herald', 'Sunrise Chronicle',
    'Prabhat Samachar', 'Nav Bharat Times', 'The Hindu Mirror', 'Daily Observer',
    'India Post', 'Deccan Star',
]

scores = []
for t in sample_titles:
    p  = preprocess(t)
    rs = top_similar(p['stripped'], [strip_affixes(x) for x in TITLES], top_k=1)
    scores.append(rs[0]['score'] if rs else 0.0)

colors = ['#ff4d6d' if s >= 80 else '#fbbf24' if s >= 55 else '#4ade80' for s in scores]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(sample_titles, scores, color=colors, edgecolor='#1e1e2e')
ax.axvline(80, color='#ff4d6d', linestyle='--', linewidth=1, label='Reject threshold (80%)')
ax.axvline(55, color='#fbbf24', linestyle='--', linewidth=1, label='Review threshold (55%)')
ax.set_xlabel('Max Similarity Score (%)')
ax.set_title('PRGI Title Similarity Scores — Sample Batch')
ax.legend()
ax.set_facecolor('#0e0e1a')
fig.patch.set_facecolor('#080810')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.title.set_color('#e8ff47')
for spine in ax.spines.values():
    spine.set_edgecolor('#1e1e2e')
plt.tight_layout()
plt.savefig('demo_scores.png', dpi=150, bbox_inches='tight')
plt.show()